# 1001 · 01 · New files arrive

The control centre drops one production report per day. This notebook plays that role: **each run adds the next two days** after the last report in the volume. Run it (**Run all**) whenever you want new data for the job.

In [ ]:
%python
import os
import shutil

CATALOG = "lab1001"

# This notebook's folder is the lab folder inside your Git folder.
LAB_PATH = os.getcwd()
if not LAB_PATH.startswith("/Workspace"):
    ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
    LAB_PATH = "/Workspace" + os.path.dirname(ctx.notebookPath().get())

print("Lab:    ", LAB_PATH)
print("Catalog:", CATALOG)

In [ ]:
%python
import csv
import random
from datetime import date, timedelta

DAYS_PER_RUN = 2
target = f"/Volumes/{CATALOG}/landing/data"

# Turbine master: (park id, park, model, rated power in MW), 12 turbines each
PARKS = [("P01", "High Ridge", "VX-4200", 4.2),
         ("P02", "Long Plains", "VX-4200", 4.2),
         ("P03", "North Cape", "VX-3000", 3.0)]


def capacity_factor(wind):
    """Simplified power curve: nothing below 3 m/s, full power at 12."""
    return min(1.0, max(0.0, (wind - 3) / 9)) ** 1.5


def write_day(day, path):
    """One synthetic report: one row per turbine, same wind pattern every time."""
    rnd = random.Random(day.toordinal())
    park_wind = {park_id: rnd.uniform(5.0, 11.0) for park_id, *_ in PARKS}
    with open(path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["date", "park_id", "park", "turbine_id", "model", "rated_power_mw",
                    "avg_wind_speed_ms", "operating_hours", "energy_mwh"])
        for park_id, park, model, rated in PARKS:
            for n in range(1, 13):
                wind = max(0.0, rnd.gauss(park_wind[park_id], 0.6))
                hours = 24.0 if rnd.random() > 0.03 else 0.0
                energy = rated * hours * capacity_factor(wind) * rnd.uniform(0.93, 1.0)
                w.writerow([day.isoformat(), park_id, park, f"{park_id}-WTG{n:02d}", model,
                            rated, round(wind, 1), hours, round(energy, 2)])


# The next days start after the last report already in the volume
existing = [f.name for f in dbutils.fs.ls(target) if f.name.startswith("production_")]
last = max(date.fromisoformat(n[len("production_"):-len(".csv")]) for n in existing)

for i in range(1, DAYS_PER_RUN + 1):
    day = last + timedelta(days=i)
    file = f"production_{day.isoformat()}.csv"
    if os.path.exists(f"{LAB_PATH}/data/{file}"):
        shutil.copyfile(f"{LAB_PATH}/data/{file}", f"{target}/{file}")
    else:
        write_day(day, f"{target}/{file}")
    print("new file:", file)

display(dbutils.fs.ls(target))

Each run adds **two more files** to the volume. The job's `COPY INTO` loads only the files it has not seen yet, so every run of the job adds exactly those two days.